In [1]:
%load_ext autoreload
%autoreload 2

import sqlite3
import pandas as pd
import numpy as np
import seaborn as sns
from scipy import stats
from datetime import datetime
from order_fingers import TouchTracker
import matplotlib.pyplot as plt
import warnings
from itertools import combinations
from sklearn.preprocessing import LabelEncoder, StandardScaler

warnings.filterwarnings('ignore')

%matplotlib inline
plt.style.use('seaborn-v0_8-darkgrid')

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

np.random.seed(42)

print("Libraries loaded successfully!")

Libraries loaded successfully!


In [2]:
DB_PATH = "./labeled_new.sqlite"

SENSOR_COLUMNS = [
    'button_pressed',
    'motor_angle',
    'touch_1_position', 'touch_1_pressure', 'touch_1_channel',
    'touch_2_position', 'touch_2_pressure', 'touch_2_channel',
    'touch_3_position', 'touch_3_pressure', 'touch_3_channel',
    'touch_4_position', 'touch_4_pressure', 'touch_4_channel',
    'touch_5_position', 'touch_5_pressure', 'touch_5_channel',
]
TIMESTAMP_COLUMN = 'timestamp_ms'
ID_COLUMNS = ['participant_id', 'session_task_id']
LABEL_COLUMN = 'parent_gesture'

WINDOW_SIZE_SAMPLES = 60
WINDOW_STRIDE_SAMPLES = 20
SAMPLING_RATE_HZ = 70  # in Hz
TOUCH_MOVE_THRESHOLD = 0.01  # Radians
WINDOW_GESTURE_MAJORITY_THRESHOLD = 0.4  # 40%

In [3]:
query = """
WITH start_end_indices AS (
    SELECT
        s.session_task_id,
        s.sensor_data_index AS start_index,
        MIN(e.sensor_data_index) AS next_end_index
    FROM preciseMarker s
    JOIN preciseMarker e 
        ON e.session_task_id = s.session_task_id
       AND e.sensor_data_index > s.sensor_data_index
    WHERE s.marker_type = 'start'
      AND e.marker_type = 'end'
    GROUP BY s.session_task_id, s.sensor_data_index
),
start_marker_timestamps AS (
    SELECT
        sensorData.session_task_id,
        session_task_row_index,
        preciseMarker.id AS group_id
    FROM sensorData
    JOIN preciseMarker
        ON sensorData.session_task_id = preciseMarker.session_task_id
       AND session_task_row_index = preciseMarker.sensor_data_index
    WHERE preciseMarker.marker_type = 'start'
),
gesture_samples AS (
    SELECT 
        sd.id,
        sd.session_task_id,
        sd.session_task_row_index,
        sd.timestamp,
        sd.timestamp_ms,
        sd.button_pressed,
        sd.motor_angle,
        sd.touch_1_position,
        sd.touch_1_pressure,
        sd.touch_1_channel,
        sd.touch_2_position,
        sd.touch_2_pressure,
        sd.touch_2_channel,
        sd.touch_3_position,
        sd.touch_3_pressure,
        sd.touch_3_channel,
        sd.touch_4_position,
        sd.touch_4_pressure,
        sd.touch_4_channel,
        sd.touch_5_position,
        sd.touch_5_pressure,
        sd.touch_5_channel,
        rt.title AS gesture,
        rtParent.title AS parent_gesture,
        s.participant_id,
        CASE WHEN rtParent.title = 'No Gesture' or smt.group_id is null THEN 0 ELSE 1 END AS is_gesture,
        smt.group_id
    FROM sensorData sd
    left JOIN start_end_indices sei
      ON sd.session_task_id = sei.session_task_id
     AND sd.session_task_row_index BETWEEN sei.start_index AND sei.next_end_index
    left JOIN start_marker_timestamps smt
      ON smt.session_task_id = sei.session_task_id
     AND smt.session_task_row_index = sei.start_index
    LEFT JOIN sessionTask st
      ON sd.session_task_id = st.id
    LEFT JOIN session s
      ON st.session_id = s.id
    LEFT JOIN recordingTask rt
      ON st.recording_task_id = rt.id
    LEFT JOIN recordingTask rtParent
      ON rt.parent_id = rtParent.id
    WHERE rtParent.title != 'Szenarien'
      AND rtParent.title != 'No Gesture'
)

-- Final combined dataset
SELECT * FROM gesture_samples
ORDER BY session_task_id, session_task_row_index;
"""

with sqlite3.connect(DB_PATH) as conn:
    df_raw = pd.read_sql_query(query, conn)

In [4]:
# Get all sensor data from markers within the 'Szenarien' parent task, though the markers there are not "start" and "end", but multiple different in the pattern "marker:xxx:start" and "marker:xxx:end"
szenarien_query = """
WITH start_end_indices AS (
    SELECT
        s.session_task_id,
        s.sensor_data_index AS start_index,
        MIN(e.sensor_data_index) AS next_end_index
    FROM preciseMarker s
    JOIN preciseMarker e 
        ON e.session_task_id = s.session_task_id
       AND e.sensor_data_index > s.sensor_data_index
    WHERE s.marker_type = 'start'
      AND e.marker_type = 'end'
    GROUP BY s.session_task_id, s.sensor_data_index
),
start_marker_timestamps AS (
    SELECT
        sensorData.session_task_id,
        session_task_row_index,
        timestamp AS group_id
    FROM sensorData
    JOIN preciseMarker
        ON sensorData.session_task_id = preciseMarker.session_task_id
       AND session_task_row_index = preciseMarker.sensor_data_index
    WHERE preciseMarker.marker_type = 'start'
),
marker_indices AS (
    SELECT
        s.session_task_id,
        s.sensor_data_index AS start_index,
        s.marker_type,
        MIN(e.sensor_data_index) AS end_index
    FROM preciseMarker s
    JOIN preciseMarker e 
        ON e.session_task_id = s.session_task_id
       AND e.sensor_data_index > s.sensor_data_index
    JOIN sessionTask st
      ON s.session_task_id = st.id
    JOIN recordingTask rt
      ON st.recording_task_id = rt.id
    JOIN recordingTask rtParent
      ON rt.parent_id = rtParent.id
    WHERE rtParent.title = 'Szenarien'
      AND s.marker_type LIKE 'marker:%:start'
      AND e.marker_type LIKE 'marker:%:end'
    GROUP BY s.session_task_id, s.sensor_data_index
)
SELECT
    sd.id,
    sd.session_task_id,
    sd.session_task_row_index,
    sd.timestamp,
    sd.button_pressed,
    sd.motor_angle,
    sd.touch_1_position,
    sd.touch_1_pressure,
    sd.touch_1_channel,
    sd.touch_2_position,
    sd.touch_2_pressure,
    sd.touch_2_channel,
    sd.touch_3_position,
    sd.touch_3_pressure,
    sd.touch_3_channel,
    sd.touch_4_position,
    sd.touch_4_pressure,
    sd.touch_4_channel,
    sd.touch_5_position,
    sd.touch_5_pressure,
    sd.touch_5_channel,
    'Szenarien' AS gesture,
    'Szenarien' AS parent_gesture,
    s.participant_id,
    CASE WHEN mi.marker_type is null or SUBSTR(mi.marker_type, 8, LENGTH(mi.marker_type) - 13) = 'No Gesture' THEN 0 ELSE 1 END AS is_gesture,
    -- Get marker type without the 'marker:' prefix and ':start' suffix
    SUBSTR(mi.marker_type, 8, LENGTH(mi.marker_type) - 13) AS marker_label,
    smt.group_id
FROM sensorData sd
JOIN start_end_indices sei
    ON sd.session_task_id = sei.session_task_id
    AND sd.session_task_row_index BETWEEN sei.start_index AND sei.next_end_index
JOIN start_marker_timestamps smt
    ON smt.session_task_id = sei.session_task_id
    AND smt.session_task_row_index = sei.start_index
LEFT JOIN marker_indices mi
  ON sd.session_task_id = mi.session_task_id
 AND sd.session_task_row_index BETWEEN mi.start_index AND mi.end_index
LEFT JOIN sessionTask st
  ON sd.session_task_id = st.id
LEFT JOIN session s
    ON st.session_id = s.id
LEFT JOIN recordingTask rt
    ON st.recording_task_id = rt.id
LEFT JOIN recordingTask rtParent
    ON rt.parent_id = rtParent.id
ORDER BY sd.session_task_id, sd.session_task_row_index;
"""
with sqlite3.connect(DB_PATH) as conn:
    szenarien_df_raw = pd.read_sql_query(szenarien_query, conn)
szenarien_df_raw.head()

In [ ]:
# print the markerlabel counts in the szenarien_df
print(szenarien_df_raw['marker_label'].value_counts())

In [ ]:
df_raw['timestamp'] = pd.to_datetime(df_raw['timestamp'])
df_raw['time_diff'] = df_raw.groupby(['session_task_id', 'timestamp']).cumcount()
df_raw['recordings_per_second'] = df_raw.groupby(['session_task_id', 'timestamp'])['id'].transform('count')
df_raw['estimated_timestamp'] = df_raw['timestamp'] + pd.to_timedelta(df_raw['time_diff'] / df_raw['recordings_per_second'], unit='s')
df_raw = df_raw.drop(columns=['time_diff', 'recordings_per_second'])

df_raw["timestamp_ms"] = (df_raw["estimated_timestamp"].astype(np.int64) // 10**6).astype(np.int64)

In [ ]:
# add timestamp columns for szenarien_df
szenarien_df_raw['timestamp'] = pd.to_datetime(szenarien_df_raw['timestamp'])
szenarien_df_raw['time_diff'] = szenarien_df_raw.groupby(['session_task_id', 'timestamp']).cumcount()
szenarien_df_raw['recordings_per_second'] = szenarien_df_raw.groupby(['session_task_id', 'timestamp'])['id'].transform('count')
szenarien_df_raw['estimated_timestamp'] = szenarien_df_raw['timestamp'] + pd.to_timedelta(szenarien_df_raw['time_diff'] / szenarien_df_raw['recordings_per_second'], unit='s')
szenarien_df_raw = szenarien_df_raw.drop(columns=['time_diff', 'recordings_per_second'])

szenarien_df_raw["timestamp_ms"] = (szenarien_df_raw["estimated_timestamp"].astype(np.int64) // 10**6).astype(np.int64)

In [ ]:
df_raw.head()

In [ ]:
df_raw.info()

In [ ]:
df_raw["gesture"] = df_raw["gesture"].astype("category")
df_raw["parent_gesture"] = df_raw["parent_gesture"].astype("category")

In [ ]:
df_raw[LABEL_COLUMN].value_counts()

In [ ]:
# Sort after ID_COLUMNS and TIMESTAMP_COLUMN
df_raw = df_raw.sort_values(by=ID_COLUMNS + [TIMESTAMP_COLUMN]).reset_index(drop=True)

In [ ]:
df_raw.groupby(["participant_id"]).size()

In [ ]:
# Check all columns with nan values
nan_columns = df_raw.columns[df_raw.isna().any()].tolist()
nan_columns

In [ ]:
# Drop rows where touch_1_position is null
initial_row_count = len(df_raw)
df_raw = df_raw.dropna(subset=['touch_1_position']).reset_index(drop=True)
dropped_row_count = initial_row_count - len(df_raw)
print(f"Dropped {dropped_row_count} rows with null touch_1_position.")

In [ ]:
import pandas as pd
import numpy as np

TOUCH_COLUMNS_BASE = ['position', 'pressure', 'channel']
MAX_TOUCHES = 5
TOUCH_PREFIXES = [f'touch_{i+1}' for i in range(MAX_TOUCHES)]

def process_and_stabilize_touches(df: pd.DataFrame):
    all_stabilized_rows = []
    active_ids_per_row = []
    
    tracker = TouchTracker() # Set your desired frame buffer
    tracker_group_id = None
    id_to_slot_map = {} 

    for index, row in df.iterrows():
        current_group_id = row.get('session_task_id', None)
        
        if tracker_group_id != current_group_id:
            tracker.clear()
            id_to_slot_map = {} 
            tracker_group_id = current_group_id
        
        # 1. Extract raw touches
        new_touches = []
        for prefix in TOUCH_PREFIXES:
            pos = row[f'{prefix}_position']
            press = row[f'{prefix}_pressure']
            chan = row[f'{prefix}_channel']
            new_touches.append((pos, press, chan))
            
        # 2. Assign IDs via tracker (includes persistence logic)
        matched_touches = tracker.assign_ids(new_touches)
        
        # 3. Assign IDs to persistent slots
        row_slots = [np.nan] * (MAX_TOUCHES * len(TOUCH_COLUMNS_BASE))
        current_row_ids = []

        for touch in matched_touches:
            touch_id, touch_data = touch[0], touch[1:]
            current_row_ids.append(touch_id)

            # Assign a slot if this is a brand new ID
            if touch_id not in id_to_slot_map:
                existing_slots = set(id_to_slot_map.values())
                for slot_idx in range(MAX_TOUCHES):
                    if slot_idx not in existing_slots:
                        id_to_slot_map[touch_id] = slot_idx
                        break
            
            slot = id_to_slot_map[touch_id]
            start_idx = slot * len(TOUCH_COLUMNS_BASE)
            row_slots[start_idx : start_idx + 3] = touch_data

        # --- 4. IMPROVED CLEANUP ---
        # Only remove from the slot map if the ID is NOT in active_touches 
        # AND NOT in the persistence buffer.
        all_remembered_ids = set(tracker.active_touches.keys()) | set(tracker.persistence_buffer.keys())
        
        # We use list() to avoid "dictionary changed size during iteration" errors
        ids_to_forget = [tid for tid in id_to_slot_map if tid not in all_remembered_ids]
        for tid in ids_to_forget:
            del id_to_slot_map[tid]

        all_stabilized_rows.append(row_slots)
        active_ids_per_row.append(current_row_ids)

    # Build the final DataFrame
    new_col_names = [f'{p}_{c}' for p in TOUCH_PREFIXES for c in TOUCH_COLUMNS_BASE]
    stabilized_df = pd.DataFrame(all_stabilized_rows, index=df.index, columns=new_col_names)
    stabilized_df['active_ids'] = active_ids_per_row
    
    return stabilized_df

In [ ]:
# Stabilize dataframe
stabilized_touch_data = process_and_stabilize_touches(df_raw.copy())
non_touch_cols = [c for c in df_raw.columns if not any(c.startswith(f'touch_{i+1}') for i in range(MAX_TOUCHES))]
final_df = df_raw[non_touch_cols].copy()

df_clean = final_df.join(stabilized_touch_data)
print("Stabilization Complete.")

In [ ]:
# Stabilize szenarien_df
stabilized_szenarien_data = process_and_stabilize_touches(szenarien_df_raw.copy())
non_touch_cols_szenarien = [c for c in szenarien_df_raw.columns if not any(c.startswith(f'touch_{i+1}') for i in range(MAX_TOUCHES))]
final_szenarien_df = szenarien_df_raw[non_touch_cols_szenarien].copy()
szenarien_df = final_szenarien_df.join(stabilized_szenarien_data)
print("Szenarien Stabilization Complete.")

In [ ]:
# Set 0 for null values in position, pressure, channel columns for touches 2 to 5
for touch_num in range(2, 6):
    position_col = f'touch_{touch_num}_position'
    pressure_col = f'touch_{touch_num}_pressure'
    channel_col = f'touch_{touch_num}_channel'
    
    df_clean[position_col] = df_clean[position_col].fillna(0)
    df_clean[pressure_col] = df_clean[pressure_col].fillna(0)
    df_clean[channel_col] = df_clean[channel_col].fillna(0)
df_clean[nan_columns].isna().sum()

In [ ]:
# Set 0 for null values in szenarien_df touch columns
for touch_num in range(1, 6):
    position_col = f'touch_{touch_num}_position'
    pressure_col = f'touch_{touch_num}_pressure'
    channel_col = f'touch_{touch_num}_channel'
    
    szenarien_df[position_col] = szenarien_df[position_col].fillna(0)
    szenarien_df[pressure_col] = szenarien_df[pressure_col].fillna(0)
    szenarien_df[channel_col] = szenarien_df[channel_col].fillna(0)
szenarien_df[nan_columns].isna().sum()

In [ ]:
# Print the min, max and mean count of samples per group_id
grouped = df_clean.groupby('group_id').size()
print(f"Min samples per group_id: {grouped.min()}")
print(f"Max samples per group_id: {grouped.max()}")
print(f"Mean samples per group_id: {grouped.mean():.2f}")

# Print the parent_gesture for the group_id with the maximum samples
max_group_id = grouped.idxmax()
max_parent_gesture = df_clean[df_clean['group_id'] == max_group_id]['parent_gesture'].iloc[0]
print(f"Parent gesture for group_id with max samples ({max_group_id}): {max_parent_gesture}")

# Print the parent_gesture for the group_id with the minimum samples
min_group_id = grouped.idxmin()
min_parent_gesture = df_clean[df_clean['group_id'] == min_group_id]['parent_gesture'].iloc[0]
print(f"Parent gesture for group_id with min samples ({min_group_id}): {min_parent_gesture}")

# Print the count of samples for the No Gesture parent_gesture that has the max samples
no_gesture_grouped = df_clean[df_clean['parent_gesture'] == 'No Gesture'].groupby('group_id').size()
print(f"Max samples for 'No Gesture' group_id: {no_gesture_grouped.max()}")

In [ ]:
# Change the WINDOW_GESTURE_MAJORITY_THRESHOLD based on the minimum samples per group_id, so that a window, that only includes the minimum amount of samples can still be labeled
WINDOW_GESTURE_MAJORITY_THRESHOLD = 1 / grouped.min()

In [ ]:
# Create sliding windows from the cleaned DataFrame
X_windows = []
y_windows = []
meta_data = []
print("Creating sliding windows...")
for (session_task_id, participant_id), group in df_clean.groupby(['session_task_id', 'participant_id']):
    sensor_data = group[SENSOR_COLUMNS].values
    labels = group[LABEL_COLUMN].values
    is_gesture = group['is_gesture'].values
    
    num_samples = sensor_data.shape[0]
    for start in range(0, num_samples - WINDOW_SIZE_SAMPLES + 1, WINDOW_STRIDE_SAMPLES):
        end = start + WINDOW_SIZE_SAMPLES
        window_data = sensor_data[start:end]
        window_labels = labels[start:end]
        window_is_gesture = is_gesture[start:end]
        
        # Majority label in the window, where as long as WINDOW_GESTURE_MAJORITY_THRESHOLD of the labels are the same and not 'No Gesture', that label is assigned
        label_counts = pd.Series(window_labels).value_counts(normalize=True)
        majority_label = 'No Gesture'
        for label, proportion in label_counts.items():
            if label != 'No Gesture' and proportion >= WINDOW_GESTURE_MAJORITY_THRESHOLD:
                majority_label = label
                break
        # Same goes for is_gesture, only if less than WINDOW_GESTURE_MAJORITY_THRESHOLD of the samples are gestures, label as no gesture
        is_gesture_counts = pd.Series(window_is_gesture).value_counts(normalize=True)
        if is_gesture_counts.get(1, 0) < WINDOW_GESTURE_MAJORITY_THRESHOLD:
            majority_label = 'No Gesture'
        
        X_windows.append(window_data)
        y_windows.append(majority_label)
        meta_data.append({
            'session_task_id': session_task_id,
            'participant_id': participant_id,
            'start_index': start,
            'end_index': end
        })
X_windows = np.array(X_windows)
y_windows = np.array(y_windows)
meta_data_df = pd.DataFrame(meta_data)
print(f"Created {X_windows.shape[0]} windows of size {WINDOW_SIZE_SAMPLES} samples.")
# Display the shape of the created windows
X_windows.shape, y_windows.shape, meta_data_df.shape


In [ ]:
# Print the value counts of the labels in the windows
unique, counts = np.unique(y_windows, return_counts=True)
label_counts = dict(zip(unique, counts))
print("Window Label Counts:")
for label, count in label_counts.items():
    print(f"{label}: {count}")

In [ ]:
# create sliding windows from szenarien_df, where the majority vote (of the marker label) determines the window label
X_windows_szenarien = []
y_windows_szenarien = []
meta_data_szenarien = []
print("Creating sliding windows for Szenarien...")
for (session_task_id, participant_id), group in szenarien_df.groupby(['session_task_id', 'participant_id']):
    sensor_data = group[SENSOR_COLUMNS].values
    labels = group['marker_label'].values
    
    num_samples = sensor_data.shape[0]
    for start in range(0, num_samples - WINDOW_SIZE_SAMPLES + 1, WINDOW_STRIDE_SAMPLES):
        end = start + WINDOW_SIZE_SAMPLES
        window_data = sensor_data[start:end]
        window_labels = labels[start:end]
        
        # Majority label in the window, where as long as WINDOW_GESTURE_MAJORITY_THRESHOLD of the labels are the same and not 'No Gesture', that label is assigned
        label_counts = pd.Series(window_labels).value_counts(normalize=True)
        majority_label = 'No Gesture'
        for label, proportion in label_counts.items():
            if label != 'No Gesture' and proportion >= WINDOW_GESTURE_MAJORITY_THRESHOLD:
                majority_label = label
                break
        # Same goes for is_gesture, only if less than WINDOW_GESTURE_MAJORITY_THRESHOLD of the samples are gestures, label as no gesture
        is_gesture_counts = pd.Series(window_labels != 'No Gesture').value_counts(normalize=True)
        if is_gesture_counts.get(True, 0) < WINDOW_GESTURE_MAJORITY_THRESHOLD:
            majority_label = 'No Gesture'
        
        X_windows_szenarien.append(window_data)
        y_windows_szenarien.append(majority_label)
        meta_data_szenarien.append({
            'session_task_id': session_task_id,
            'participant_id': participant_id,
            'start_index': start,
            'end_index': end
        })
X_windows_szenarien = np.array(X_windows_szenarien)
y_windows_szenarien = np.array(y_windows_szenarien)
meta_data_szenarien_df = pd.DataFrame(meta_data_szenarien)
print(f"Created {X_windows_szenarien.shape[0]} windows of size {WINDOW_SIZE_SAMPLES} samples for Szenarien.")
# Display the shape of the created windows for szenarien
X_windows_szenarien.shape, y_windows_szenarien.shape, meta_data_szenarien_df.shape

In [ ]:
# Print the value counts of the labels in the szenarien windows
unique_szenarien, counts_szenarien = np.unique(y_windows_szenarien, return_counts=True)
label_counts_szenarien = dict(zip(unique_szenarien, counts_szenarien))
print("Szenarien Window Label Counts:")
for label, count in label_counts_szenarien.items():
    print(f"{label}: {count}")

In [ ]:
def extract_features(window, is_padded_window):
    features = []
    feature_names = [] # for debug purposes we store the names of each feature at the same index
    
    # Assume sensor column order from SENSOR_COLUMNS
    # Indices: button=0, motor=1, touch_pos=[2,5,8,11,14], touch_press=[3,6,9,12,15]
    button_col = 0
    motor_col = 1
    touch_channel_cols = [4, 7, 10, 13, 16]  # touch_X_channel indices
    touch_pressure_cols = [3, 6, 9, 12, 15]  # touch_X_pressure indices
    touch_position_cols = [2, 5, 8, 11, 14]  # touch_X_position indices
    original_length_col = len(SENSOR_COLUMNS)  # last column is original length

    button_press_data = window[:, button_col]
    motor_data = window[:, motor_col]
    touch_pressures = window[:, touch_pressure_cols]
    touch_positions = window[:, touch_position_cols]
    # Only consider touch positions where the pressure > 0
    valid_mask = touch_pressures > 0
    touch_positions = touch_positions[valid_mask]
    original_length = int(window[0, original_length_col]) if is_padded_window else window.shape[0]

    # A 1 signalizes the button was pressed, this can happen multiple times in a window, so we first have to extract each consecutive time a button was pressed, plus the amount of samples for each time
    binary_button_press = (button_press_data > 0).astype(int)
    padded_button_press = np.concatenate(([0], binary_button_press, [0]))
    button_press_diff = np.diff(padded_button_press)
    button_press_starts = np.where(button_press_diff == 1)[0]
    button_press_ends = np.where(button_press_diff == -1)[0]
    button_press_durations = button_press_ends - button_press_starts

    def circular_range(angles):
        # 1. Sort the angles
        sorted_angles = np.sort(angles)
        # 2. Calculate gaps between neighbors
        gaps = np.diff(sorted_angles)
        # 3. Calculate the wrap-around gap (last to first)
        wrap_gap = (2 * np.pi - sorted_angles[-1]) + sorted_angles[0]
        # 4. Combine all gaps
        all_gaps = np.append(gaps, wrap_gap)
        # 5. Range is the full circle minus the biggest empty space
        return 2 * np.pi - np.max(all_gaps)
    
    def add_feature(name, value):
        features.append(value)
        feature_names.append(name)

    # Button Features
    # 1. Button pressed at all?
    add_feature("button_pressed_any", np.any(binary_button_press))
    # # 2. How often was the button pressed (changed from 0 to 1 and vice versa) and how often was it let go
    add_feature("button_press_count", len(button_press_starts))
    # # 3. Mean (raw + durations)
    add_feature("button_press_mean", np.mean(binary_button_press))
    add_feature("button_press_duration_mean", np.mean(button_press_durations) if len(button_press_durations) > 0 else 0)
    # # 4. Std Dev (raw + durations)
    # add_feature("button_press_std", np.std(binary_button_press))
    # add_feature("button_press_duration_std", np.std(button_press_durations) if len(button_press_durations) > 0 else 0)
    # # 5. Median (raw + durations)
    add_feature("button_press_median", np.median(binary_button_press))
    add_feature("button_press_duration_median", np.median(button_press_durations) if len(button_press_durations) > 0 else 0)
    # # 6. Max duration
    # add_feature("button_press_max_duration", np.max(button_press_durations) if len(button_press_durations) > 0 else 0)
    # # 7. Min duration
    # add_feature("button_press_min_duration", np.min(button_press_durations) if len(button_press_durations) > 0 else 0)
    # # 8. Range duration
    # add_feature("button_press_range_duration", circular_range(button_press_durations) if len(button_press_durations) > 0 else 0)
    # # 9. Total duration
    # add_feature("button_press_total_duration", np.sum(button_press_durations) if len(button_press_durations) > 0 else 0)

    # Motor Angle Features
    # 1. Mean
    add_feature("motor_angle_mean", np.mean(motor_data))
    # # 2. Std Dev
    add_feature("motor_angle_std", np.std(motor_data))
    # # 3. Min Max
    add_feature("motor_angle_min", np.min(motor_data))
    add_feature("motor_angle_max", np.max(motor_data))
    # # 4. Range
    add_feature("motor_angle_range", circular_range(motor_data))
    # # 5. Median
    add_feature("motor_angle_median", np.median(motor_data))
    # # 6. Q25 / Q75
    # add_feature("motor_angle_q25", np.percentile(motor_data, 25))
    # add_feature("motor_angle_q75", np.percentile(motor_data, 75))
    # # 7. Skewness / Kurtosis
    # add_feature("motor_angle_skewness", stats.skew(motor_data))
    # add_feature("motor_angle_kurtosis", stats.kurtosis(motor_data))
    # # 8. Velocity
    # if len(motor_data) > 1:
        # motor_velocity = np.diff(motor_data) * SAMPLING_RATE_HZ
        # add_feature("motor_velocity_mean", np.mean(motor_velocity))
        # add_feature("motor_velocity_std", np.std(motor_velocity))
        # add_feature("motor_velocity_max", np.max(motor_velocity))
    # else:
        # add_feature("motor_velocity_mean", 0)
        # add_feature("motor_velocity_std", 0)
        # add_feature("motor_velocity_max", 0)

    # General Touch Activity Features
    # 1. Max simultaneous touches
    active_touches = (touch_pressures > 0).astype(int)
    add_feature("max_simultaneous_touches", np.max(np.sum(active_touches, axis=1)))
    # 2. Avg active touches
    add_feature("avg_active_touches", np.mean(np.sum(active_touches, axis=1)))
    # 3. Any touch at all?
    add_feature("any_touch", np.any(active_touches))
    # 4. Total pressure across all sensors
    add_feature("total_active_touch_samples", np.sum(active_touches))
    add_feature("total_pressure", np.sum(touch_pressures))
    # 5. Max pressure
    add_feature("max_pressure", np.max(touch_pressures))
    # 5.1 Min non-zero pressure
    non_zero_pressures = touch_pressures[touch_pressures > 0]
    add_feature("min_non_zero_pressure", np.min(non_zero_pressures) if len(non_zero_pressures) > 0 else 0)
    # 6. Touch centroid
    touch_position_sin_mean = np.sin(touch_positions).mean()
    touch_position_cos_mean = np.cos(touch_positions).mean()
    touch_position_mean_angle = np.arctan2(touch_position_sin_mean, touch_position_cos_mean)
    add_feature("touch_centroid", touch_position_mean_angle % (2 * np.pi))
    # 7. Touch Spread
    touch_position_R = np.sqrt(touch_position_sin_mean**2 + touch_position_cos_mean**2)
    add_feature("touch_spread", 1 - touch_position_R)
    add_feature("touch_spread_log", -np.log(np.clip(touch_position_R, 1e-9, 1.0)))

    # Distances between each finger (including wrapping)
    def short_dist(a, b):
        return np.abs(np.arctan2(np.sin(a - b), np.cos(a - b)))
    
    i = 0
    for p1, p2 in combinations(touch_position_cols, 2):
        i += 1
        finger1_positions = window[:, p1]
        finger2_positions = window[:, p2]
        # Only consider positions where the pressure > 0
        finger1_pressures = window[:, touch_pressure_cols[touch_position_cols.index(p1)]]
        finger2_pressures = window[:, touch_pressure_cols[touch_position_cols.index(p2)]]
        valid_mask = (finger1_pressures > 0) & (finger2_pressures > 0)
        finger1_positions = finger1_positions[valid_mask]
        finger2_positions = finger2_positions[valid_mask]

        distances = short_dist(finger1_positions, finger2_positions)

        # Calculate the distances for the first and last third of the window
        third_length = len(distances) // 3
        first_third_distances = distances[:third_length]
        last_third_distances = distances[-third_length:]
        add_feature("finger_distance_mean_first_third_" + str(i), np.mean(first_third_distances) if len(first_third_distances) > 0 else 0)
        add_feature("finger_distance_mean_last_third_" + str(i), np.mean(last_third_distances) if len(last_third_distances) > 0 else 0)

        # If the distances are empty add 0 for all
        has_distance = len(distances) != 0
        add_feature("finger_distance_mean_" + str(i), np.mean(distances) if has_distance else 0)
        add_feature("finger_distance_std_" + str(i), np.std(distances) if has_distance else 0)
        add_feature("finger_distance_max_" + str(i), np.max(distances) if has_distance else 0)
        add_feature("finger_distance_min_" + str(i), np.min(distances) if has_distance else 0)
        # add_feature("finger_distance_range_" + str(i), circular_range(distances) if has_distance else 0)
        add_feature("finger_distance_q25_" + str(i), np.percentile(distances, 25) if has_distance else 0)
        add_feature("finger_distance_q75_" + str(i), np.percentile(distances, 75) if has_distance else 0)


    amount_active_moving_fingers = 0
    # Per Finger Features
    for i, (pos_col, press_col, channel_col) in enumerate(zip(touch_position_cols, touch_pressure_cols, touch_channel_cols)):
        finger_positions = window[:, pos_col]
        finger_pressures = window[:, press_col]
        finger_channels = window[:, channel_col]

        # Fill nan with 0
        finger_positions = np.nan_to_num(finger_positions, nan=0.0)
        finger_pressures = np.nan_to_num(finger_pressures, nan=0.0)

        # Only consider positions where the pressure > 0
        valid_mask = finger_pressures > 0
        finger_positions = finger_positions[valid_mask]
        finger_channels = finger_channels[valid_mask]

        # General
        # 1. Any touch activity?
        is_touching = finger_pressures > 0
        add_feature(f"finger_{i+1}_any_touch", np.any(is_touching))
        # 3. Has moved? (this has a threshold to avoid noise)
        circle_range = circular_range(finger_positions) if len(finger_positions) > 0 else 0
        has_moved = circle_range > TOUCH_MOVE_THRESHOLD
        add_feature(f"finger_{i+1}_has_moved", has_moved)

        # 4. Amount of reactivations (so how often was the finger lifted and put down again)
        touch_reactivations = 0
        for t in range(1, len(is_touching)):
            if is_touching[t] and not is_touching[t - 1]:
                touch_reactivations += 1
        add_feature(f"finger_{i+1}_touch_reactivations", touch_reactivations)

        # Get the time between reactivations
        touch_reactivation_times = []
        last_touch_end = None
        for t in range(1, len(is_touching)):
            if is_touching[t] and not is_touching[t - 1]:
                if last_touch_end is not None:
                    touch_reactivation_times.append(t - last_touch_end)
            if not is_touching[t] and is_touching[t - 1]:
                last_touch_end = t
        # Mean time between reactivations
        add_feature(f"finger_{i+1}_mean_time_between_reactivations", np.mean(touch_reactivation_times) if len(touch_reactivation_times) > 0 else 0)
        # Std Dev time between reactivations
        add_feature(f"finger_{i+1}_std_time_between_reactivations", np.std(touch_reactivation_times) if len(touch_reactivation_times) > 0 else 0)
        # Max time between reactivations
        add_feature(f"finger_{i+1}_max_time_between_reactivations", np.max(touch_reactivation_times) if len(touch_reactivation_times) > 0 else 0)
        # Min time between reactivations
        add_feature(f"finger_{i+1}_min_time_between_reactivations", np.min(touch_reactivation_times) if len(touch_reactivation_times) > 0 else 0)
        # Range time between reactivations
        # add_feature(f"finger_{i+1}_range_time_between_reactivations", np.abs(np.max(touch_reactivation_times) - np.min(touch_reactivation_times)) if len(touch_reactivation_times) > 0 else 0)

        # Touch Times and Amount (Pressure is 0 or null if no touch) There might be multiple touches in a window for a given finger, so we have to extract each consecutive time a touch was active, plus the amount of samples for each time.
        # start_of_touches = is_touching & (~is_touching.shift(1, fill_value=False))
        # end_of_touches = is_touching & (~is_touching.shift(-1, fill_value=False))
        start_of_touches = is_touching & ~np.insert(is_touching[:-1], 0, False)
        end_of_touches = is_touching & ~np.append(is_touching[1:], False)
        touch_starts = np.where(start_of_touches)[0]
        touch_ends = np.where(end_of_touches)[0]
        touch_durations = touch_ends - touch_starts

        max_touch_duration = np.max(touch_durations) if len(touch_durations) > 0 else 0
        min_touch_duration = np.min(touch_durations) if len(touch_durations) > 0 else 0
        
        # Is continuously pressed during the window? (If the min touch duration is the length of the window)
        is_continuous_touch = min_touch_duration >= original_length - 1
        add_feature(f"finger_{i+1}_continuous_touch_duration", min_touch_duration if is_continuous_touch else 0)
        add_feature(f"finger_{i+1}_non_continuous_touch_duration", 0 if is_continuous_touch else np.sum(touch_durations) if len(touch_durations) > 0 else 0)
        # add_feature(f"finger_{i+1}_total_touch_duration", np.sum(touch_durations) if len(touch_durations) > 0 else 0)
        add_feature(f"finger_{i+1}_max_touch_duration", max_touch_duration)
        add_feature(f"finger_{i+1}_min_touch_duration", min_touch_duration)
        add_feature(f"finger_{i+1}_mean_touch_duration", np.mean(touch_durations) if len(touch_durations) > 0 else 0)
        # add_feature(f"finger_{i+1}_std_touch_duration", np.std(touch_durations) if len(touch_durations) > 0 else 0)
        add_feature(f"finger_{i+1}_is_continuous_touch", is_continuous_touch)

        if np.any(is_touching) and (has_moved or not is_continuous_touch):
            amount_active_moving_fingers += 1

        # 4. How many times was the finger touched?
        add_feature(f"finger_{i+1}_touch_starts", len(touch_starts))
        add_feature(f"finger_{i+1}_touch_ends", len(touch_ends))
        # 5. Mean touch duration
        add_feature(f"finger_{i+1}_mean_touch_duration", np.mean(touch_durations) if len(touch_durations) > 0 else 0)
        # 6. Std Dev touch duration
        add_feature(f"finger_{i+1}_std_touch_duration", np.std(touch_durations) if len(touch_durations) > 0 else 0)
        # 7. Max touch duration
        add_feature(f"finger_{i+1}_max_touch_duration", max_touch_duration)
        # 8. Min touch duration
        add_feature(f"finger_{i+1}_min_touch_duration", min_touch_duration)
        # 9. Range touch duration
        add_feature(f"finger_{i+1}_range_touch_duration", np.abs(max_touch_duration - min_touch_duration) if len(touch_durations) > 0 else 0)
        # 10. Total touch duration
        # add_feature(f"finger_{i+1}_total_touch_duration", np.sum(touch_durations) if len(touch_durations) > 0 else 0)

        # Position, Pressure Features
        for data_type, data in [("positions", finger_positions), ("pressures", finger_pressures), ("channels", finger_channels)]:
            # If the data is empty, add zeros for all features
            has_data = len(data) != 0
            # 1. Mean pressure
            add_feature(f"finger_{i+1}_{data_type}_mean", np.mean(data) if has_data else 0)
            # 2. Std Dev pressure
            add_feature(f"finger_{i+1}_{data_type}_std", np.std(data) if has_data else 0)
            # 3. Max pressure
            add_feature(f"finger_{i+1}_{data_type}_max", np.max(data) if has_data else 0)
            # 4. Min pressure
            add_feature(f"finger_{i+1}_{data_type}_min", np.min(data) if has_data else 0)
            # 5. Range pressure
            add_feature(f"finger_{i+1}_{data_type}_range", circular_range(data) if has_data else 0)
            # 7. Q25 / Q75
            add_feature(f"finger_{i+1}_{data_type}_q25", np.percentile(data, 25) if has_data else 0)
            add_feature(f"finger_{i+1}_{data_type}_q75", np.percentile(data, 75) if has_data else 0)
            # 8. Skewness / Kurtosis
            add_feature(f"finger_{i+1}_{data_type}_skewness", stats.skew(data) if has_data else 0)
            add_feature(f"finger_{i+1}_{data_type}_kurtosis", stats.kurtosis(data) if has_data else 0)
            if data_type == "channels":
                continue
            # 9. Velocity
            has_velocity = len(data) > 1
            pressure_velocity = np.diff(data) * SAMPLING_RATE_HZ
            add_feature(f"finger_{i+1}_{data_type}_velocity_mean", np.mean(pressure_velocity) if has_velocity else 0)
            add_feature(f"finger_{i+1}_{data_type}_velocity_std", np.std(pressure_velocity) if has_velocity else 0)
            add_feature(f"finger_{i+1}_{data_type}_velocity_max", np.max(pressure_velocity) if has_velocity else 0)
            if data_type == "pressures":
                continue
            # 10. Acceleration
            has_acceleration = len(data) > 2
            pressure_acceleration = np.diff(data, n=2) * (SAMPLING_RATE_HZ ** 2)
            add_feature(f"finger_{i+1}_{data_type}_acceleration_mean", np.mean(pressure_acceleration) if has_acceleration else 0)
            add_feature(f"finger_{i+1}_{data_type}_acceleration_std", np.std(pressure_acceleration) if has_acceleration else 0)
            add_feature(f"finger_{i+1}_{data_type}_acceleration_max", np.max(pressure_acceleration) if has_acceleration else 0)

    # 8. Amount of fingers minus the amount of fingers that are constantly touching and not moving
    add_feature("active_moving_fingers", amount_active_moving_fingers)

    return np.array(features, dtype=np.float32), feature_names

In [ ]:
le = LabelEncoder()
y_windows_encoded = le.fit_transform(y_windows)
y_windows_szenarien_encoded = le.transform(y_windows_szenarien)

In [ ]:
# Print nan for both datasets before feature extraction
print(f"NaN in windows before feature extraction: {np.isnan(X_windows).any()}")
print(f"NaN in scenario windows before feature extraction: {np.isnan(X_windows_szenarien).any()}")

In [ ]:
feature_names = extract_features(X_windows[0], False)[1]
# Extract the Features for all windows (ignoring feature names)
X_windows_extracted = np.array([extract_features(win, False)[0] for win in X_windows], dtype=np.float32)
print(f"Extracted features shape: {X_windows_extracted.shape}")  # should be (N_windows, N_features)

X_windows_szenarien_extracted = np.array([extract_features(seq, False)[0] for seq in X_windows_szenarien], dtype=np.float32)
print(f"Extracted scenario features shape: {X_windows_szenarien_extracted.shape}")  # should be (N_gestures, N_features)

In [ ]:
# Print nan for both datasets after feature extraction
print(f"NaN in windows after feature extraction: {np.isnan(X_windows_extracted).any()}")
print(f"NaN in scenario windows after feature extraction: {np.isnan(X_windows_szenarien_extracted).any()}")

In [ ]:
# Set all nan values to 0
X_win = np.nan_to_num(X_windows_extracted, nan=0.0)
X_win_seq = np.nan_to_num(X_windows_szenarien_extracted, nan=0.0)

In [ ]:
# Lets visualize the X_sequences for the "Double Tap" gesture
gesture_name = "Single Tap"
print("Values for gesture:", gesture_name)
gesture_index = np.where(le.classes_ == gesture_name)[0][0]
indices = np.where(y_windows_encoded == gesture_index)[0]
if len(indices) == 0:
    print(f"No samples found for gesture: {gesture_name}")
else:
    sample_index = indices[0]
    sample_features = X_win[sample_index]

    # Print the values with the feature name for this row
    for fname, fvalue in zip(feature_names, sample_features):
        print(f"{fname}: {fvalue}")

In [ ]:
# Draw a graph of the raw finger position data for the same sample
raw_sample = df_clean[df_clean["session_task_id"] == meta_data_df.iloc[indices[0]]["session_task_id"]]
plt.figure(figsize=(12, 6))
for i in range(5):
    pos_col = f'touch_{i+1}_position'
    plt.plot(raw_sample['estimated_timestamp'], raw_sample[pos_col], label=f'Touch {i+1} Position')
plt.xlabel('Time')
plt.ylabel('Finger Position (normalized)')
plt.title(f'Raw Finger Positions for Gesture: {gesture_name}')
plt.legend()
plt.show()

In [ ]:
# Draw a graph of the finger positions for all windows that was classified as the gesture, as if they where performed right after one another, with vertical colored lines indicating the window boundaries
plt.figure(figsize=(12, 6))
for i in range(5):
    pos_col = f'touch_{i+1}_position'
    combined_positions = []
    for idx in indices:
        combined_positions.extend(X_windows[idx, :, 2 + i * 3])  # touch_X_position index in SENSOR_COLUMNS
    plt.plot(combined_positions, label=f'Touch {i+1} Position')
plt.xlabel('Combined Time')
plt.ylabel('Finger Position (normalized)')
plt.title(f'Combined Finger Positions for Gesture: {gesture_name}')
# Draw vertical lines for window boundaries
for w in range(1, len(indices)):
    plt.axvline(x=w * WINDOW_SIZE_SAMPLES, color='red', linestyle='--', alpha=0.5)
plt.xlim(650, 1800)
plt.legend()
plt.show()

In [ ]:
participant_ids = meta_data_df['participant_id'].unique().tolist()
amount_of_participants = len(participant_ids)

train_pids = participant_ids[:amount_of_participants - 2]
val_pids = participant_ids[amount_of_participants - 2:amount_of_participants - 1]
test_pids = participant_ids[amount_of_participants - 1:]

print(f"Amount of participants: {amount_of_participants} (Train: {len(train_pids)}, Val: {len(val_pids)}, Test: {len(test_pids)})")

In [ ]:
# Split sliding windows
train_mask_win = meta_data_df['participant_id'].isin(train_pids)
X_train_win = X_win[train_mask_win]
y_train_win = y_windows_encoded[train_mask_win]

val_mask_win = meta_data_df['participant_id'].isin(val_pids)
X_val_win = X_win[val_mask_win]
y_val_win = y_windows_encoded[val_mask_win]

test_mask_win = meta_data_df['participant_id'].isin(test_pids)
X_test_win = X_win[test_mask_win]
y_test_win = y_windows_encoded[test_mask_win]

In [ ]:
# For each label, print the amount of samples in each split
for label in le.classes_:
    label_int = le.transform([label])[0]
    train_count = np.sum(y_train_win == label_int)
    val_count = np.sum(y_val_win == label_int)
    test_count = np.sum(y_test_win == label_int)
    print(f"Label: {label} | Train: {train_count} | Val: {val_count} | Test: {test_count}")

In [ ]:
# Print train / val / test shapes for windows for both full and windowed
print(f"Sliding Window Split:")
print(f"  X_train_win: {X_train_win.shape}, y_train_win: {y_train_win.shape}")
print(f"  X_val_win:   {X_val_win.shape},   y_val_win:   {y_val_win.shape}")
print(f"  X_test_win:  {X_test_win.shape},  y_test_win:  {y_test_win.shape}")

In [ ]:
# Scale both datasets
scaler = StandardScaler()
X_train_win = scaler.fit_transform(X_train_win)
X_val_win = scaler.transform(X_val_win)
X_test_win = scaler.transform(X_test_win)

# scale scenario datasets
X_test_szenarien = scaler.transform(X_windows_szenarien_extracted)

In [ ]:
# CV Training
# from sklearn.ensemble import RandomForestClassifier
# from sklearn.neighbors import KNeighborsClassifier
# from sklearn.preprocessing import StandardScaler
# from sklearn.ensemble import ExtraTreesClassifier
# from sklearn.svm import LinearSVC
# from sklearn.preprocessing import StandardScaler
# from sklearn.pipeline import Pipeline
# from sklearn.ensemble import AdaBoostClassifier
# from sklearn.naive_bayes import GaussianNB

# MODEL_CONFIGS = [
#     {
#         "name": "RandomForest",
#         "pipeline": Pipeline([
#             ("clf", RandomForestClassifier(random_state=42, n_jobs=-1))
#         ]),
#         "param_grid": {
#             "clf__n_estimators":      [50, 200, 400, 800],
#             "clf__max_depth":         [None, 10, 20, 30],
#             "clf__min_samples_split": [2, 5, 10],
#             "clf__min_samples_leaf":  [1, 2, 4],
#             "clf__max_features":      ["sqrt", "log2", 0.5],
#             "clf__class_weight":      [None, "balanced"],
#         },
#     },
#     {
#         "name": "KNN",
#         "pipeline": Pipeline([
#             ("scaler", StandardScaler()),
#             ("clf", KNeighborsClassifier())
#         ]),
#         "param_grid": {
#             "clf__n_neighbors": [3, 5, 9],
#             "clf__weights": ["uniform", "distance"],
#             "clf__p": [1, 2],  # 1 = Manhattan, 2 = Euclidean
#         },
#     },
#     {
#         "name": "ExtraTrees",
#         "pipeline": Pipeline([
#             ("clf", ExtraTreesClassifier(random_state=42, n_jobs=-1))
#         ]),
#         "param_grid": {
#             "clf__n_estimators":      [50, 200, 400, 800],
#             "clf__max_depth":         [None, 10, 20, 30],
#             "clf__min_samples_split": [2, 5, 10],
#             "clf__min_samples_leaf":  [1, 2, 4],
#             "clf__max_features":      ["sqrt", "log2", 0.5],
#             "clf__class_weight":      [None, "balanced"],
#         },
#     },
#     {
#         "name": "LinearSVC",
#         "pipeline": Pipeline([
#             ("scaler", StandardScaler()),
#             ("clf", LinearSVC(
#                 max_iter=5000,
#                 dual=False
#             ))
#         ]),
#         "param_grid": {
#             "clf__C": [0.1, 1.0, 10.0],
#             "clf__class_weight": [None, "balanced"],
#         },
#     },
#     {
#         "name": "AdaBoost",
#         "pipeline": Pipeline([
#             ("clf", AdaBoostClassifier(random_state=42))
#         ]),
#         "param_grid": {
#             "clf__n_estimators": [50, 100, 200],
#             "clf__learning_rate": [0.01, 0.1, 1.0],
#         },
#     },
#     {
#         "name": "GaussianNB",
#         "pipeline": Pipeline([
#             ("scaler", StandardScaler()),
#             ("clf", GaussianNB())
#         ]),
#         "param_grid": {
#             # No hyperparameters to tune for GaussianNB
#         },
#     }
# ]


In [ ]:
# CV Training
# from sklearn.model_selection import GroupKFold, GridSearchCV

# SCORING = "f1_macro"
# gkf = GroupKFold(n_splits=amount_of_participants-2)
# groups_train = meta_data_df[train_mask_win]["participant_id"].values

# best_model = None
# best_model_name = None
# best_score = -np.inf
# all_results = []

# for cfg in MODEL_CONFIGS:
#     name = cfg["name"]
#     pipe = cfg["pipeline"]
#     param_grid = cfg["param_grid"]

#     print(f"\n=== Tuning {name} ===")
#     grid = GridSearchCV(
#         estimator=pipe,
#         param_grid=param_grid,
#         cv=gkf,
#         scoring=SCORING,
#         n_jobs=-1,
#         verbose=2
#     )
#     grid.fit(X_train_win, y_train_win, groups=groups_train)

#     print(f"{name} best params: {grid.best_params_}")
#     print(f"{name} best CV {SCORING}: {grid.best_score_:.4f}")

#     all_results.append({
#         "model": name,
#         "best_score": grid.best_score_,
#         "best_params": grid.best_params_,
#         "estimator": grid.best_estimator_,
#     })

#     if grid.best_score_ > best_score:
#         best_score = grid.best_score_
#         best_model = grid.best_estimator_
#         best_model_name = name

# print("\n=== Summary over all models ===")
# for res in all_results:
#     print(f"{res['model']}: {SCORING}={res['best_score']:.4f}")

# print(f"\nBest overall model: {best_model_name} with {SCORING}={best_score:.4f}")


In [ ]:
# Random Training
from scipy.stats import randint, uniform, loguniform
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

MODEL_CONFIGS = [
    {
        "name": "RandomForest",
        "pipeline": Pipeline([
            ("clf", RandomForestClassifier(random_state=42, n_jobs=-1))
        ]),
        "param_grid": {
            "clf__n_estimators": randint(50, 800), # Sample any integer between 50-800
            "clf__max_depth": [None, 5, 10, 30, 50],
            "clf__min_samples_split": randint(2, 11),
            "clf__min_samples_leaf": randint(1, 5),
            "clf__max_features": ["sqrt", "log2", 0.5],
            "clf__class_weight": [None, "balanced"],
        },
    },
    {
        "name": "KNN",
        "pipeline": Pipeline([
            ("scaler", StandardScaler()),
            ("clf", KNeighborsClassifier())
        ]),
        "param_grid": {
            "clf__n_neighbors": randint(3, 15),
            "clf__weights": ["uniform", "distance"],
            "clf__p": [1, 2],
        },
    },
    {
        "name": "ExtraTrees",
        "pipeline": Pipeline([
            ("clf", ExtraTreesClassifier(random_state=42, n_jobs=-1))
        ]),
        "param_grid": {
            "clf__n_estimators": randint(50, 800),
            "clf__max_depth": [None, 5, 10, 30, 50],
            "clf__min_samples_split": randint(2, 11),
            "clf__min_samples_leaf": randint(1, 5),
            "clf__max_features": ["sqrt", "log2", 0.5],
            "clf__class_weight": [None, "balanced"],
        },
    }
]

In [ ]:
# Random Training
from sklearn.model_selection import GroupKFold, RandomizedSearchCV

SCORING = "f1_macro"
gkf = GroupKFold(n_splits=amount_of_participants-2)
groups_train = meta_data_df[train_mask_win]["participant_id"].values

best_model = None
best_model_name = None
best_score = -np.inf
all_results = []

for cfg in MODEL_CONFIGS:
    name = cfg["name"]
    pipe = cfg["pipeline"]
    param_grid = cfg["param_grid"]

    print(f"\n=== Tuning {name} with RandomizedSearch ===")
    
    # RandomizedSearchCV picks a fixed number of settings (n_iter)
    # rather than trying every single combination.
    search = RandomizedSearchCV(
        estimator=pipe,
        param_distributions=param_grid, # Note: grid becomes distributions
        n_iter=30,                     # Adjust this! Higher = better, but slower
        cv=gkf,
        scoring=SCORING,
        n_jobs=-1,
        verbose=1,                     # Reduced verbosity slightly
        random_state=42                # Essential for reproducibility
    )
    
    search.fit(X_train_win, y_train_win, groups=groups_train)

    print(f"{name} best params: {search.best_params_}")
    print(f"{name} best CV {SCORING}: {search.best_score_:.4f}")

    all_results.append({
        "model": name,
        "best_score": search.best_score_,
        "best_params": search.best_params_,
        "estimator": search.best_estimator_,
    })

    if search.best_score_ > best_score:
        best_score = search.best_score_
        best_model = search.best_estimator_
        best_model_name = name

print("\n=== Summary over all models ===")
for res in all_results:
    print(f"{res['model']}: {SCORING}={res['best_score']:.4f}")

print(f"\nBest overall model: {best_model_name} with {SCORING}={best_score:.4f}")


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

y_test_pred = best_model.predict(X_test_win)

# print("Test classification report:\n",
#       classification_report(y_test_win, y_test_pred, target_names=le.classes_))

cm = confusion_matrix(y_test_win, y_test_pred)
cm_norm = cm.astype("float") / cm.sum(axis=1, keepdims=True)

plt.figure(figsize=(8, 6))
sns.heatmap(cm_norm, annot=True, fmt=".2f",
            xticklabels=le.classes_,
            yticklabels=le.classes_,
            cmap="Blues")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Normalized Confusion Matrix (Test)")
plt.tight_layout()
plt.show()

In [ ]:
# Print which features are the most influencial for the best model if possible
if hasattr(best_model.named_steps['clf'], 'feature_importances_'):
    importances = best_model.named_steps['clf'].feature_importances_
    feature_importance_pairs = sorted(zip(feature_names, importances), key=lambda x: x[1], reverse=True)

    print("\nTop 50 Feature Importances:")
    for fname, importance in feature_importance_pairs[:50]:
        print(f"{fname}: {importance:.4f}")

In [ ]:
# Drain a DNN on the same data for comparison
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

def create_dnn_model(input_shape, num_classes):
    model = keras.Sequential([
        layers.InputLayer(input_shape=input_shape),
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(64, activation='relu'),
        layers.Dropout(0.1),
        layers.Dense(num_classes, activation="softmax")
    ])

    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

dnn_model = create_dnn_model(input_shape=(X_train_win.shape[1],), num_classes=len(le.classes_))
dnn_model.summary()
early_stopping = keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)
history = dnn_model.fit(
    X_train_win, y_train_win,
    validation_data=(X_val_win, y_val_win),
    epochs=100,
    batch_size=32,
    callbacks=[early_stopping],
    verbose=2
)
# Evaluate on test set
test_loss, test_accuracy = dnn_model.evaluate(X_test_win, y_test_win, verbose=0)
print(f"DNN Test Accuracy: {test_accuracy:.4f}")

y_test_dnn_pred_probs = dnn_model.predict(X_test_win)
y_test_dnn_pred = np.argmax(y_test_dnn_pred_probs, axis=1)
print("DNN Test classification report:\n",
      classification_report(y_test_win, y_test_dnn_pred, target_names=le.classes_))

cm_dnn = confusion_matrix(y_test_win, y_test_dnn_pred)
cm_dnn_norm = cm_dnn.astype("float") / cm_dnn.sum(axis=1, keepdims=True)
plt.figure(figsize=(8, 6))
sns.heatmap(cm_dnn_norm, annot=True, fmt=".2f",
            xticklabels=le.classes_,
            yticklabels=le.classes_,
            cmap="Greens")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("DNN Normalized Confusion Matrix (Test)")
plt.tight_layout()
plt.show()
        

In [ ]:
# compare the confusion matrixes and show a plot comparing both models
plt.figure(figsize=(16, 6))
plt.subplot(1, 2, 1)
sns.heatmap(cm_norm, annot=True, fmt=".2f",
            xticklabels=le.classes_,
            yticklabels=le.classes_,
            cmap="Blues")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("RF Normalized Confusion Matrix (Test)")
plt.subplot(1, 2, 2)
sns.heatmap(cm_dnn_norm, annot=True, fmt=".2f",
            xticklabels=le.classes_,
            yticklabels=le.classes_,
            cmap="Greens")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("DNN Normalized Confusion Matrix (Test)")
plt.tight_layout()
plt.show()

In [ ]:
# Run the best model with the szenarien windows and print classification report
y_szenarien_pred = best_model.predict(X_win_seq)
print("Szenarien classification report:\n",
      classification_report(y_windows_szenarien_encoded, y_szenarien_pred, target_names=le.classes_))

cm_szenarien = confusion_matrix(y_windows_szenarien_encoded, y_szenarien_pred)
cm_szenarien_norm = cm_szenarien.astype("float") / cm_szenarien.sum(axis=1, keepdims=True)
plt.figure(figsize=(8, 6))
sns.heatmap(cm_szenarien_norm, annot=True, fmt=".2f",
            xticklabels=le.classes_,
            yticklabels=le.classes_,
            cmap="Purples")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Normalized Confusion Matrix (Szenarien)")
plt.tight_layout()
plt.show()

In [ ]:
# run the DNN model with the szenarien windows and print classification report
y_szenarien_dnn_pred_probs = dnn_model.predict(X_win_seq)
y_szenarien_dnn_pred = np.argmax(y_szenarien_dnn_pred_probs, axis=1)
print("Szenarien DNN classification report:\n",
      classification_report(y_windows_szenarien_encoded, y_szenarien_dnn_pred, target_names=le.classes_))
cm_szenarien_dnn = confusion_matrix(y_windows_szenarien_encoded, y_szenarien_dnn_pred)
cm_szenarien_dnn_norm = cm_szenarien_dnn.astype("float") / cm_szenarien_dnn.sum(axis=1, keepdims=True)
plt.figure(figsize=(8, 6))
sns.heatmap(cm_szenarien_dnn_norm, annot=True, fmt=".2f",
            xticklabels=le.classes_,
            yticklabels=le.classes_,
            cmap="Oranges")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("DNN Normalized Confusion Matrix (Szenarien)")
plt.tight_layout()
plt.show()

In [ ]:
# compare the confusion matrixes and show a plot comparing both models for szenarien
plt.figure(figsize=(16, 6))
plt.subplot(1, 2, 1)
sns.heatmap(cm_szenarien_norm, annot=True, fmt=".2f",
            xticklabels=le.classes_,
            yticklabels=le.classes_,
            cmap="Purples")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("RF Normalized Confusion Matrix (Szenarien)")
plt.subplot(1, 2, 2)
sns.heatmap(cm_szenarien_dnn_norm, annot=True, fmt=".2f",
            xticklabels=le.classes_,
            yticklabels=le.classes_,
            cmap="Oranges")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("DNN Normalized Confusion Matrix (Szenarien)")
plt.tight_layout()
plt.show()